<div style="background: linear-gradient(135deg, #0f0c29, #302b63, #24243e); padding: 40px 30px; border-radius: 16px; margin-bottom: 20px;">
<h1 style="text-align:center; color:#e17055; font-family:Verdana; letter-spacing:3px;">
?? Image Matching Challenge 2022 ? Feature Matching Pipeline
</h1>
<h4 style="text-align:center; color:#dfe6e9; font-family:Verdana;">
Notebook 2 of 2 ? Cumulative Feature Matching & Evaluation
</h4>
<hr style="border:1px solid #6c5ce7;">
<p style="color:#fab1a0; font-family:Verdana; font-size:14px; text-align:center;">
This notebook implements our production-style image matching pipeline for IMC 2022. It combines preprocessing, feature extraction, matching, robust estimation, and mAA evaluation into a single Kaggle-ready workflow with clear stage boundaries and measurable outputs.
</p>
</div>

## ?? Table of Contents

1. [Environment Setup & Imports](#s1)
2. [Data Preprocessing Pipeline](#s2)
3. [Feature Extraction Methods](#s3)
  - 3.1 SIFT Extraction
  - 3.2 ORB Extraction
  - 3.3 LoFTR (Detector-Free, Kornia)
4. [Feature Matching Methods](#s4)
  - 4.1 Brute-Force + Ratio Test
  - 4.2 FLANN + Ratio Test
  - 4.3 LoFTR Dense Matching
5. [Robust Estimation ? RANSAC Variants](#s5)
  - 5.1 Standard RANSAC
  - 5.2 LMEDS
  - 5.3 USAC MAGSAC++
6. [Fundamental Matrix & Pose Recovery](#s6)
7. [Evaluation ? mAA Metric](#s7)
8. [Pipeline Comparison, Inference & Submission](#s8)
  - 8.1 Scene-Level Runtime Metrics
  - 8.2 Test Inference Helpers (SIFT + LoFTR)
  - 8.3 Submission CSV Export


<a id="s1"></a>
## 1 · Environment Setup & Imports

In [ ]:
import os, gc, csv, time, random, warnings, math
from collections import namedtuple
import numpy as np
import pandas as pd
import cv2
import matplotlib
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

warnings.filterwarnings("ignore")

# Standard matplotlib style for Kaggle compatibility
plt.style.use('default')

random.seed(42); np.random.seed(42)
eps = 1e-15
SRC = "/kaggle/input/image-matching-challenge-2022"
Gt = namedtuple("Gt", ["K", "R", "T"])

# Try importing kornia/torch for LoFTR
try:
  import torch
  import kornia as K_lib
  import kornia.feature as KF
  HAS_KORNIA = True
  DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
  print(f"? Kornia available ? device: {DEVICE}")
except ImportError:
  HAS_KORNIA = False
  print("?? Kornia not available ? LoFTR sections will be skipped")

print(f"? OpenCV: {cv2.__version__}")
print(f"? Data dir: {SRC}")


<a id="s2"></a>
## 2 ? Data Preprocessing Pipeline

Production preprocessing steps used throughout the pipeline:
- **Resize**: Scale longest edge to 840px for speed/quality balance
- **Grayscale**: Convert image views for classical detector stability
- **Covisibility filter**: Keep pairs ? 0.1 to prioritize useful overlap
- **Calibration parsing**: Load K, R, T matrices for geometric recovery

In [ ]:
# ── Data loading ──
train_csv_path = os.path.join(SRC, "train/scaling_factors.csv")
scaling_df = pd.read_csv(train_csv_path)
scale_map = scaling_df.groupby("scene")["scaling_factor"].first().to_dict()
SCENES = scaling_df.scene.unique().tolist()

# ── Helper functions ──
def load_calibration(path):
  calib = {}
  df = pd.read_csv(path)
  for _, row in df.iterrows():
    iid = row["image_id"]
    K = np.fromstring(row["camera_intrinsics"], sep=" ").reshape(3, 3)
    R = np.fromstring(row["rotation_matrix"], sep=" ").reshape(3, 3)
    T = np.fromstring(row["translation_vector"], sep=" ").reshape(3,)
    calib[iid] = Gt(K=K, R=R, T=T)
  return calib

def read_covisibility(path):
  cov = {}
  df = pd.read_csv(path)
  for _, row in df.iterrows():
    cov[row["pair"]] = row["covisibility"]
  return cov, df

def preprocess_image(img, max_edge=840):
  scale = max_edge / max(img.shape[0], img.shape[1])
  w, h = int(img.shape[1] * scale), int(img.shape[0] * scale)
  return cv2.resize(img, (w, h)), scale

def load_and_preprocess(path, grayscale=False, max_edge=840):
  flag = cv2.IMREAD_GRAYSCALE if grayscale else cv2.IMREAD_COLOR
  img = cv2.imread(path, flag)
  if img is None:
    raise FileNotFoundError(f"Cannot load: {path}")
  img_resized, scale = preprocess_image(img, max_edge)
  return img_resized, scale

def load_torch_image(path, device, max_edge=840):
  img = cv2.imread(path)
  scale = max_edge / max(img.shape[0], img.shape[1])
  w, h = int(img.shape[1] * scale), int(img.shape[0] * scale)
  img = cv2.resize(img, (w, h))
  img_t = K_lib.image_to_tensor(img, False).float() / 255.0
  img_t = K_lib.color.bgr_to_rgb(img_t)
  return img_t.to(device), scale

# ── Quaternion & error helpers ──
def quaternion_from_matrix(R):
  t = np.trace(R)
  if t > 0:
    s = 0.5 / math.sqrt(t + 1.0)
    w = 0.25 / s
    x = (R[2,1] - R[1,2]) * s
    y = (R[0,2] - R[2,0]) * s
    z = (R[1,0] - R[0,1]) * s
  elif R[0,0] > R[1,1] and R[0,0] > R[2,2]:
    s = 2.0 * math.sqrt(1.0 + R[0,0] - R[1,1] - R[2,2])
    w = (R[2,1] - R[1,2]) / s
    x = 0.25 * s
    y = (R[0,1] + R[1,0]) / s
    z = (R[0,2] + R[2,0]) / s
  elif R[1,1] > R[2,2]:
    s = 2.0 * math.sqrt(1.0 + R[1,1] - R[0,0] - R[2,2])
    w = (R[0,2] - R[2,0]) / s
    x = (R[0,1] + R[1,0]) / s
    y = 0.25 * s
    z = (R[1,2] + R[2,1]) / s
  else:
    s = 2.0 * math.sqrt(1.0 + R[2,2] - R[0,0] - R[1,1])
    w = (R[1,0] - R[0,1]) / s
    x = (R[0,2] + R[2,0]) / s
    y = (R[1,2] + R[2,1]) / s
    z = 0.25 * s
  return np.array([w, x, y, z])

def compute_error(q_gt, T_gt, q_pred, T_pred, scale):
  q_gt = q_gt / (np.linalg.norm(q_gt) + eps)
  q_pred = q_pred / (np.linalg.norm(q_pred) + eps)
  d = abs(np.dot(q_gt, q_pred))
  d = min(1.0, d)
  err_q = 2 * math.acos(d) * 180 / math.pi

  T_gt_scaled = T_gt * scale
  T_pred_norm = T_pred / (np.linalg.norm(T_pred) + eps) * np.linalg.norm(T_gt_scaled)
  err_t = min(np.linalg.norm(T_gt_scaled - T_pred_norm),
        np.linalg.norm(T_gt_scaled + T_pred_norm))
  return err_q, err_t

print("✅ Preprocessing pipeline ready")
print(f"  Scenes: {len(SCENES)}")

# ?? Additional utilities from n2 pipelines ??

def decode_fundamental(f_matrix_str):
  vals = [float(v) for v in f_matrix_str.strip().split()]
  if len(vals) != 9:
    raise ValueError("Fundamental matrix string must contain 9 values")
  return np.array(vals, dtype=np.float64).reshape(3, 3)


def encode_fundamental(F):
  return np.array(F, dtype=np.float64).reshape(9,)


def FlattenMatrix(M, num_digits=8):
  return ' '.join([f'{v:.{num_digits}e}' for v in np.asarray(M).reshape(-1)])


def array_from_cv_kps(kps):
  return np.array([kp.pt for kp in kps], dtype=np.float32)


def normalize_keypoints(keypoints, K):
  cx, cy = K[0, 2], K[1, 2]
  fx, fy = K[0, 0], K[1, 1]
  return (keypoints - np.array([[cx, cy]], dtype=np.float64)) / np.array([[fx, fy]], dtype=np.float64)


def compute_essential_matrix(F, K1, K2, kp1, kp2):
  if F is None or np.asarray(F).shape != (3, 3):
    raise ValueError("Malformed fundamental matrix")
  E = (K2.T @ F @ K1).astype(np.float64)
  kp1n = normalize_keypoints(np.asarray(kp1, dtype=np.float64), K1)
  kp2n = normalize_keypoints(np.asarray(kp2, dtype=np.float64), K2)
  _, R, T, _ = cv2.recoverPose(E, kp1n, kp2n)
  return E, R, T



### Covisibility per Scene — Filtering Threshold

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(20, 16))
total_pairs = 0
for idx, scene in enumerate(SCENES):
  ax = axes[idx // 4][idx % 4]
  _, pco_df = read_covisibility(f"{SRC}/train/{scene}/pair_covisibility.csv")
  vals = pco_df["covisibility"].values
  total_pairs += len(vals)
  above = (vals >= 0.1).sum()
  ax.hist(vals, bins=40, color="#6c5ce7", edgecolor="#00cec9", alpha=0.8)
  ax.axvline(0.1, color="#e17055", lw=2, ls="--")
  ax.set_title(f"{scene.replace('_',' ').title()} ({above}/{len(vals)})",
         fontsize=9, color="#fdcb6e")
  ax.set_xlabel("cov", fontsize=7)
fig.suptitle(f"Covisibility Distributions — Total pairs: {total_pairs:,}",
       fontsize=14, color="#00cec9", y=1.01)
plt.tight_layout(); plt.show()

<a id="s3"></a>
## 3 · Feature Extraction Methods

### 3.1 SIFT Extraction on Real Dataset Pair

In [ ]:
# Pick a real pair from Sacre Coeur (high covisibility)
SCENE_DEMO = "sacre_coeur"
_, pco_demo = read_covisibility(f"{SRC}/train/{SCENE_DEMO}/pair_covisibility.csv")
pco_demo = pco_demo[pco_demo.covisibility >= 0.3].sort_values("covisibility", ascending=False)
demo_pair = pco_demo.iloc[0]
ids = demo_pair["pair"].split("-")
P1 = f"{SRC}/train/{SCENE_DEMO}/images/{ids[0]}.jpg"
P2 = f"{SRC}/train/{SCENE_DEMO}/images/{ids[1]}.jpg"

img1_pre, s1 = load_and_preprocess(P1, grayscale=False)
img2_pre, s2 = load_and_preprocess(P2, grayscale=False)
g1 = cv2.cvtColor(img1_pre, cv2.COLOR_BGR2GRAY)
g2 = cv2.cvtColor(img2_pre, cv2.COLOR_BGR2GRAY)
rgb1 = cv2.cvtColor(img1_pre, cv2.COLOR_BGR2RGB)
rgb2 = cv2.cvtColor(img2_pre, cv2.COLOR_BGR2RGB)

# Show the pair
fig, axes = plt.subplots(1, 2, figsize=(20, 8))
axes[0].imshow(rgb1); axes[0].set_title(f"{SCENE_DEMO} — Image 1\n{ids[0]}", color="#00cec9")
axes[1].imshow(rgb2); axes[1].set_title(f"{SCENE_DEMO} — Image 2\n{ids[1]}", color="#00cec9")
for ax in axes: ax.axis("off")
fig.suptitle(f"Demo Pair · Covisibility = {demo_pair.covisibility:.4f}", fontsize=14, color="#fdcb6e")
plt.tight_layout(); plt.show()

# SIFT
NUM_FEATURES = 5000
sift = cv2.SIFT_create(NUM_FEATURES, contrastThreshold=-10000, edgeThreshold=-10000)
t0 = time.time()
kp1_s, des1_s = sift.detectAndCompute(g1, None)
kp2_s, des2_s = sift.detectAndCompute(g2, None)
sift_t = time.time() - t0

vis1 = cv2.drawKeypoints(rgb1, kp1_s, None, color=(0,255,200),
  flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
vis2 = cv2.drawKeypoints(rgb2, kp2_s, None, color=(0,255,200),
  flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)

fig, axes = plt.subplots(1, 2, figsize=(22, 8))
axes[0].imshow(vis1); axes[0].set_title(f"SIFT · {len(kp1_s)} keypoints", color="#00cec9")
axes[1].imshow(vis2); axes[1].set_title(f"SIFT · {len(kp2_s)} keypoints", color="#00cec9")
for ax in axes: ax.axis("off")
fig.suptitle(f"SIFT Feature Extraction — {sift_t:.3f}s", fontsize=14, color="#fdcb6e")
plt.tight_layout(); plt.show()
print(f"📊 SIFT: {len(kp1_s)} + {len(kp2_s)} kps | desc: {des1_s.shape} | {sift_t:.3f}s")

### 3.2 ORB Extraction

In [ ]:
orb = cv2.ORB_create(nfeatures=NUM_FEATURES)
t0 = time.time()
kp1_o, des1_o = orb.detectAndCompute(g1, None)
kp2_o, des2_o = orb.detectAndCompute(g2, None)
orb_t = time.time() - t0

vis1o = cv2.drawKeypoints(rgb1, kp1_o, None, color=(255,100,50), flags=0)
vis2o = cv2.drawKeypoints(rgb2, kp2_o, None, color=(255,100,50), flags=0)

fig, axes = plt.subplots(1, 2, figsize=(22, 8))
axes[0].imshow(vis1o); axes[0].set_title(f"ORB · {len(kp1_o)} keypoints", color="#e17055")
axes[1].imshow(vis2o); axes[1].set_title(f"ORB · {len(kp2_o)} keypoints", color="#e17055")
for ax in axes: ax.axis("off")
fig.suptitle(f"ORB Feature Extraction — {orb_t:.3f}s", fontsize=14, color="#fdcb6e")
plt.tight_layout(); plt.show()
print(f"📊 ORB: {len(kp1_o)} + {len(kp2_o)} kps | desc: {des1_o.shape} | {orb_t:.3f}s")

### 3.3 LoFTR (Detector-Free Matching via Kornia)

In [ ]:
if HAS_KORNIA:
  # Load LoFTR
  matcher_loftr = KF.LoFTR(pretrained=None)
  ckpt_path = "/kaggle/input/kornia-loftr/loftr_outdoor.ckpt"
  if os.path.exists(ckpt_path):
    matcher_loftr.load_state_dict(torch.load(ckpt_path)["state_dict"])
  else:
    matcher_loftr = KF.LoFTR(pretrained="outdoor")
  matcher_loftr = matcher_loftr.to(DEVICE).eval()

  img1_t, _ = load_torch_image(P1, DEVICE)
  img2_t, _ = load_torch_image(P2, DEVICE)

  t0 = time.time()
  with torch.no_grad():
    inp = {"image0": K_lib.color.rgb_to_grayscale(img1_t),
        "image1": K_lib.color.rgb_to_grayscale(img2_t)}
    corr = matcher_loftr(inp)
  loftr_t = time.time() - t0

  mkpts0 = corr["keypoints0"].cpu().numpy()
  mkpts1 = corr["keypoints1"].cpu().numpy()
  conf = corr["confidence"].cpu().numpy()

  print(f"📊 LoFTR: {len(mkpts0)} correspondences | {loftr_t:.3f}s")
  print(f"  Confidence — mean: {conf.mean():.3f}, min: {conf.min():.3f}, max: {conf.max():.3f}")

  # Visualize LoFTR matches (high confidence)
  high_conf = conf > 0.5
  fig, axes = plt.subplots(1, 2, figsize=(22, 8))
  axes[0].imshow(K_lib.tensor_to_image(img1_t))
  axes[0].scatter(mkpts0[high_conf, 0], mkpts0[high_conf, 1], s=3, c='lime', alpha=0.5)
  axes[0].set_title(f"LoFTR Keypoints Img1 ({high_conf.sum()} conf>0.5)", color="#6c5ce7")
  axes[1].imshow(K_lib.tensor_to_image(img2_t))
  axes[1].scatter(mkpts1[high_conf, 0], mkpts1[high_conf, 1], s=3, c='lime', alpha=0.5)
  axes[1].set_title(f"LoFTR Keypoints Img2 ({high_conf.sum()} conf>0.5)", color="#6c5ce7")
  for ax in axes: ax.axis("off")
  fig.suptitle(f"LoFTR Dense Matching — {len(mkpts0)} total, {loftr_t:.3f}s", fontsize=14, color="#fdcb6e")
  plt.tight_layout(); plt.show()
else:
  print("⚠️ Skipping LoFTR — Kornia not installed")

<a id="s4"></a>
## 4 · Feature Matching Methods

### 4.1 Brute-Force + Lowe Ratio Test (SIFT)

In [ ]:
bf = cv2.BFMatcher(cv2.NORM_L2)
raw = bf.knnMatch(des1_s, des2_s, k=2)
good_bf = [m for m, n in raw if m.distance < 0.75 * n.distance]
good_bf = sorted(good_bf, key=lambda x: x.distance)

match_vis = cv2.drawMatches(rgb1, kp1_s, rgb2, kp2_s, good_bf[:100], None,
  matchColor=(0,255,200), singlePointColor=(255,100,50),
  flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)

fig, ax = plt.subplots(figsize=(24, 10))
ax.imshow(match_vis)
ax.set_title(f"BF + Ratio Test — {len(good_bf)} matches (top 100 shown)",
       fontsize=14, color="#00cec9")
ax.axis("off"); plt.tight_layout(); plt.show()

bf_dists = [m.distance for m in good_bf]
print(f"📊 BF Matching:")
print(f"  Total good: {len(good_bf)} / {len(raw)} raw")
print(f"  Distance — mean: {np.mean(bf_dists):.2f}, min: {np.min(bf_dists):.2f}, max: {np.max(bf_dists):.2f}")

### 4.2 FLANN + Ratio Test (SIFT)

In [ ]:
FLANN_INDEX_KDTREE = 1
index_params = dict(algorithm=FLANN_INDEX_KDTREE, trees=5)
search_params = dict(checks=50)
flann = cv2.FlannBasedMatcher(index_params, search_params)
raw_fl = flann.knnMatch(des1_s, des2_s, k=2)
good_fl = [m for m, n in raw_fl if m.distance < 0.75 * n.distance]
good_fl = sorted(good_fl, key=lambda x: x.distance)

match_vis_fl = cv2.drawMatches(rgb1, kp1_s, rgb2, kp2_s, good_fl[:100], None,
  matchColor=(100,200,255), singlePointColor=(255,100,50),
  flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)

fig, ax = plt.subplots(figsize=(24, 10))
ax.imshow(match_vis_fl)
ax.set_title(f"FLANN + Ratio Test — {len(good_fl)} matches (top 100 shown)",
       fontsize=14, color="#74b9ff")
ax.axis("off"); plt.tight_layout(); plt.show()

fl_dists = [m.distance for m in good_fl]
print(f"📊 FLANN Matching:")
print(f"  Total good: {len(good_fl)} / {len(raw_fl)} raw")
print(f"  Distance — mean: {np.mean(fl_dists):.2f}, min: {np.min(fl_dists):.2f}")

### 4.3 LoFTR Dense Matching + Confidence Filtering

In [ ]:
if HAS_KORNIA:
  thresholds = [0.3, 0.5, 0.7, 0.9]
  counts = []
  for th in thresholds:
    mask = conf >= th
    counts.append(mask.sum())

  fig, ax = plt.subplots(figsize=(7, 4))
  ax.bar([f">={th}" for th in thresholds], counts, color="mediumpurple", edgecolor="black")
  ax.set_title("LoFTR Matches by Confidence Threshold")
  ax.set_ylabel("# Matches")
  plt.tight_layout()
  plt.show()

  high = conf >= 0.5
  print(f"?? LoFTR at conf>=0.5: {high.sum()} matches")
  print(f"   Mean confidence: {conf[high].mean():.4f}")
else:
  print("?? Skipping")


<a id="s5"></a>
## 5 · Robust Estimation — RANSAC Variants

Comparing RANSAC, LMEDS, and USAC_MAGSAC++ on the SIFT BF matches.

In [ ]:
pts1_bf = np.float32([kp1_s[m.queryIdx].pt for m in good_bf])
pts2_bf = np.float32([kp2_s[m.trainIdx].pt for m in good_bf])

ransac_methods = [
  ("FM_RANSAC", cv2.FM_RANSAC, {"ransacReprojThreshold": 3.0, "confidence": 0.99}),
  ("FM_LMEDS", cv2.FM_LMEDS, {"confidence": 0.99}),
  ("USAC_MAGSAC", cv2.USAC_MAGSAC, {"ransacReprojThreshold": 0.25, "confidence": 0.99999, "maxIters": 10000}),
]

results = {}
for name, method, params in ransac_methods:
  t0 = time.time()
  F_est, mask_r = cv2.findFundamentalMat(pts1_bf, pts2_bf, method, **params)
  t_elapsed = time.time() - t0
  inlier_mask = mask_r.ravel().astype(bool) if mask_r is not None else np.zeros(len(pts1_bf), dtype=bool)
  n_inliers = inlier_mask.sum()

  inlier_matches = [m for m, flag in zip(good_bf, inlier_mask) if flag]
  results[name] = {"F": F_est, "inliers": n_inliers, "total": len(good_bf),
           "ratio": n_inliers/len(good_bf)*100, "time": t_elapsed,
           "matches": inlier_matches, "mask": inlier_mask}

  # Visualize
  vis = cv2.drawMatches(rgb1, kp1_s, rgb2, kp2_s, inlier_matches[:80], None,
    matchColor=(0,255,100), singlePointColor=None,
    flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
  fig, ax = plt.subplots(figsize=(22, 9))
  ax.imshow(vis)
  ax.set_title(f"{name} — {n_inliers} inliers / {len(good_bf)} total ({n_inliers/len(good_bf)*100:.1f}%) · {t_elapsed:.3f}s",
         fontsize=13, color="#00b894")
  ax.axis("off"); plt.tight_layout(); plt.show()

  print(f"📊 {name}: {n_inliers} inliers ({n_inliers/len(good_bf)*100:.1f}%) | {t_elapsed:.4f}s")
  print(f"  F = {F_est.ravel()[:5]}...\n")

In [ ]:
# Comparison chart
comp_df = pd.DataFrame([
  {"Method": k, "Inliers": v["inliers"], "Ratio (%)": round(v["ratio"],1), "Time (s)": round(v["time"],4)}
  for k, v in results.items()
])
print("
?? RANSAC Variant Comparison:")
print(comp_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(comp_df["Method"], comp_df["Inliers"], color=["tomato", "gold", "seagreen"], edgecolor="black")
ax.set_title("RANSAC Variants ? Inlier Count")
ax.set_ylabel("Inliers")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(comp_df["Method"], comp_df["Time (s)"], color=["tomato", "gold", "seagreen"], edgecolor="black")
ax.set_title("RANSAC Variants ? Runtime")
ax.set_ylabel("Time (s)")
plt.tight_layout()
plt.show()


<a id="s6"></a>
## 6 · Fundamental Matrix & Pose Recovery

In [ ]:
# Load calibration for our demo pair
calib = load_calibration(f"{SRC}/train/{SCENE_DEMO}/calibration.csv")
K1_cal, K2_cal = calib[ids[0]].K, calib[ids[1]].K

# Use MAGSAC result
F_best = results["USAC_MAGSAC"]["F"]
mask_best = results["USAC_MAGSAC"]["mask"]
pts1_in = pts1_bf[mask_best]
pts2_in = pts2_bf[mask_best]

# Essential matrix
E = K2_cal.T @ F_best @ K1_cal
print(f"📊 Essential Matrix:\n{E}")

# Recover pose
_, R_est, T_est, pose_mask = cv2.recoverPose(E, pts1_in, pts2_in, K1_cal)
q_est = quaternion_from_matrix(R_est)
T_est = T_est.flatten()

# Ground truth
R1_gt, T1_gt = calib[ids[0]].R, calib[ids[0]].T.reshape(3, 1)
R2_gt, T2_gt = calib[ids[1]].R, calib[ids[1]].T.reshape(3, 1)
dR_gt = R2_gt @ R1_gt.T
dT_gt = (T2_gt - dR_gt @ T1_gt).flatten()
q_gt = quaternion_from_matrix(dR_gt)

# Compute errors
err_q, err_t = compute_error(q_gt, dT_gt, q_est, T_est, scale_map[SCENE_DEMO])
print(f"\n📊 Pose Error:")
print(f"  Rotation error:  {err_q:.2f}°")
print(f"  Translation error: {err_t:.2f} m")

# Epipolar lines visualization
lines = cv2.computeCorrespondEpilines(pts2_in[:20].reshape(-1,1,2), 2, F_best).reshape(-1,3)
epi_img = rgb1.copy()
for line, pt in zip(lines, pts1_in[:20]):
  c = tuple(np.random.randint(100, 255, 3).tolist())
  x0 = 0; y0 = int(-line[2] / line[1])
  x1 = rgb1.shape[1]; y1 = int(-(line[2] + line[0] * x1) / line[1])
  epi_img = cv2.line(epi_img, (x0, y0), (x1, y1), c, 2)
  epi_img = cv2.circle(epi_img, tuple(np.int32(pt)), 6, c, -1)

fig, ax = plt.subplots(figsize=(18, 8))
ax.imshow(epi_img)
ax.set_title(f"Epipolar Lines — Rot err: {err_q:.2f}° | Trans err: {err_t:.2f}m",
       fontsize=14, color="#00cec9")
ax.axis("off"); plt.tight_layout(); plt.show()

<a id="s7"></a>
## 7 · Evaluation — mAA Metric

The competition metric: **mean Average Accuracy (mAA)** over 10 rotation & translation thresholds.

In [ ]:
thresholds_q = np.linspace(1, 10, 10)
thresholds_t = np.geomspace(0.2, 5, 10)

def evaluate_pipeline(scene, detector_fn, matcher_fn, ransac_method, ransac_params,
           max_pairs=50, num_features=5000):
  calib = load_calibration(f"{SRC}/train/{scene}/calibration.csv")
  cov_dict, cov_df = read_covisibility(f"{SRC}/train/{scene}/pair_covisibility.csv")
  pairs = [p for p, c in cov_dict.items() if c >= 0.1]
  random.shuffle(pairs)
  pairs = pairs[:max_pairs]

  errors_list = []
  match_counts, inlier_counts = [], []

  for pair_str in tqdm(pairs, desc=f" {scene}", leave=False):
    id1, id2 = pair_str.split("-")
    p1 = f"{SRC}/train/{scene}/images/{id1}.jpg"
    p2 = f"{SRC}/train/{scene}/images/{id2}.jpg"

    img1, _ = load_and_preprocess(p1, grayscale=True)
    img2, _ = load_and_preprocess(p2, grayscale=True)

    kp1, d1 = detector_fn(img1)
    kp2, d2 = detector_fn(img2)
    if d1 is None or d2 is None or len(d1) < 8 or len(d2) < 8:
      errors_list.append((180, 100)); continue

    matches = matcher_fn(d1, d2)
    if len(matches) < 8:
      errors_list.append((180, 100)); continue

    p1_pts = np.float32([kp1[m.queryIdx].pt for m in matches])
    p2_pts = np.float32([kp2[m.trainIdx].pt for m in matches])
    match_counts.append(len(matches))

    F, msk = cv2.findFundamentalMat(p1_pts, p2_pts, ransac_method, **ransac_params)
    if F is None or F.shape != (3,3):
      errors_list.append((180, 100)); continue

    inl = msk.ravel().astype(bool) if msk is not None else np.ones(len(p1_pts), bool)
    inlier_counts.append(inl.sum())

    K1, K2 = calib[id1].K, calib[id2].K
    E = K2.T @ F @ K1
    try:
      _, R_r, T_r, _ = cv2.recoverPose(E, p1_pts[inl], p2_pts[inl], K1)
    except:
      errors_list.append((180, 100)); continue

    q_pred = quaternion_from_matrix(R_r)
    R1g, T1g = calib[id1].R, calib[id1].T.reshape(3,1)
    R2g, T2g = calib[id2].R, calib[id2].T.reshape(3,1)
    dR = R2g @ R1g.T; dT = (T2g - dR @ T1g).flatten()
    q_g = quaternion_from_matrix(dR)

    eq, et = compute_error(q_g, dT, q_pred, T_r.flatten(), scale_map[scene])
    errors_list.append((eq, et))

  # Compute mAA
  errs = np.array(errors_list)
  acc = np.zeros((len(thresholds_q), len(thresholds_t)))
  for i, tq in enumerate(thresholds_q):
    for j, tt in enumerate(thresholds_t):
      acc[i, j] = ((errs[:, 0] <= tq) & (errs[:, 1] <= tt)).mean()
  mAA_val = acc.mean()

  return mAA_val, np.mean(match_counts) if match_counts else 0, np.mean(inlier_counts) if inlier_counts else 0

# Define SIFT detector + BF matcher
def sift_detect(img, nf=5000):
  det = cv2.SIFT_create(nf, contrastThreshold=-10000, edgeThreshold=-10000)
  return det.detectAndCompute(img, None)

def bf_ratio_match(d1, d2, ratio=0.75):
  bf = cv2.BFMatcher(cv2.NORM_L2)
  raw = bf.knnMatch(d1, d2, k=2)
  return sorted([m for m, n in raw if m.distance < ratio * n.distance], key=lambda x: x.distance)

print("✅ Evaluation framework ready")

In [ ]:
# Evaluate SIFT + BF + MAGSAC on all scenes
print("\n🚀 Evaluating SIFT + BF + MAGSAC++ pipeline...\n")

scene_results = {}
for scene in tqdm(SCENES, desc="Scenes"):
  maa, avg_matches, avg_inliers = evaluate_pipeline(
    scene, sift_detect, bf_ratio_match,
    cv2.USAC_MAGSAC, {"ransacReprojThreshold": 0.25, "confidence": 0.99999, "maxIters": 10000},
    max_pairs=30
  )
  scene_results[scene] = {"mAA": round(maa, 4), "avg_matches": round(avg_matches),
              "avg_inliers": round(avg_inliers)}
  print(f" {scene}: mAA={maa:.4f} | matches={avg_matches:.0f} | inliers={avg_inliers:.0f}")

overall_mAA = np.mean([v["mAA"] for v in scene_results.values()])
print(f"\n🏆 Overall mAA: {overall_mAA:.4f}")

<a id="s8"></a>
## 8 ? Pipeline Comparison, Inference & Submission

### 8.1 Scene-Level Runtime Metrics
This section reports metrics computed during execution (no static benchmark values).


In [ ]:
res_df = pd.DataFrame([
  {"Scene": k, **v} for k, v in scene_results.items()
])

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(res_df["Scene"], res_df["mAA"], color="teal", edgecolor="black")
ax.set_title(f"mAA per Scene ? SIFT + BF + MAGSAC++ (Overall: {overall_mAA:.4f})")
ax.set_ylabel("mAA")
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

x = np.arange(len(res_df))
w = 0.38
fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(x - w/2, res_df["avg_matches"], width=w, label="Avg Matches", color="mediumpurple", edgecolor="black")
ax.bar(x + w/2, res_df["avg_inliers"], width=w, label="Avg Inliers", color="seagreen", edgecolor="black")
ax.set_title("Match & Inlier Counts per Scene")
ax.set_xticks(x)
ax.set_xticklabels(res_df["Scene"], rotation=45)
ax.legend()
plt.tight_layout()
plt.show()

print("
?? Final Results Summary:")
print(res_df.to_string(index=False))
print(f"
?? Overall mAA: {overall_mAA:.4f}")


### 8.2 Test Inference Helpers (SIFT + LoFTR)

The following helpers integrate both n2 notebook inference styles:
- Classical SIFT + ratio test + USAC_MAGSAC
- LoFTR + USAC_MAGSAC (optional, when Kornia is available)

These functions are intended for Kaggle test-time inference and avoid hardcoded score assumptions.


In [ ]:
def estimate_f_single_pair_ratio(
  img1_bgr,
  img2_bgr,
  num_features=5000,
  ratio=0.85,
  ransac_thresh=0.25,
  confidence=0.99999,
  max_iters=10000,
):
  g1 = cv2.cvtColor(img1_bgr, cv2.COLOR_BGR2GRAY)
  g2 = cv2.cvtColor(img2_bgr, cv2.COLOR_BGR2GRAY)

  sift = cv2.SIFT_create(num_features, contrastThreshold=-10000, edgeThreshold=-10000)
  kp1, d1 = sift.detectAndCompute(g1, None)
  kp2, d2 = sift.detectAndCompute(g2, None)

  if d1 is None or d2 is None or len(d1) < 8 or len(d2) < 8:
    return np.zeros((3, 3), dtype=np.float64), 0

  bf = cv2.BFMatcher()
  raw = bf.knnMatch(d1, d2, k=2)
  good = [m for m, n in raw if m.distance < ratio * n.distance]

  if len(good) < 8:
    return np.zeros((3, 3), dtype=np.float64), len(good)

  pts1 = np.float32([kp1[m.queryIdx].pt for m in good])
  pts2 = np.float32([kp2[m.trainIdx].pt for m in good])

  F, msk = cv2.findFundamentalMat(
    pts1,
    pts2,
    cv2.USAC_MAGSAC,
    ransacReprojThreshold=ransac_thresh,
    confidence=confidence,
    maxIters=max_iters,
  )

  if F is None or np.asarray(F).shape != (3, 3):
    return np.zeros((3, 3), dtype=np.float64), len(good)

  return F.astype(np.float64), len(good)


def ensure_loftr_matcher(device=DEVICE):
  if not HAS_KORNIA:
    return None
  matcher = KF.LoFTR(pretrained=None)
  ckpt_path = "/kaggle/input/kornia-loftr/loftr_outdoor.ckpt"
  if os.path.exists(ckpt_path):
    matcher.load_state_dict(torch.load(ckpt_path)["state_dict"])
  else:
    matcher = KF.LoFTR(pretrained="outdoor")
  return matcher.to(device).eval()


def estimate_f_single_pair_loftr(
  image1_path,
  image2_path,
  matcher,
  conf_threshold=0.0,
  ransac_thresh=0.1845,
  confidence=0.999999,
  max_iters=220000,
):
  if matcher is None:
    return np.zeros((3, 3), dtype=np.float64), 0

  img1_t, _ = load_torch_image(image1_path, DEVICE)
  img2_t, _ = load_torch_image(image2_path, DEVICE)

  with torch.no_grad():
    corr = matcher({
      "image0": K_lib.color.rgb_to_grayscale(img1_t),
      "image1": K_lib.color.rgb_to_grayscale(img2_t),
    })

  mkpts0 = corr["keypoints0"].cpu().numpy()
  mkpts1 = corr["keypoints1"].cpu().numpy()
  conf = corr["confidence"].cpu().numpy()

  if conf_threshold > 0:
    keep = conf >= conf_threshold
    mkpts0, mkpts1 = mkpts0[keep], mkpts1[keep]

  if len(mkpts0) < 8:
    return np.zeros((3, 3), dtype=np.float64), len(mkpts0)

  F, inliers = cv2.findFundamentalMat(
    mkpts0,
    mkpts1,
    cv2.USAC_MAGSAC,
    ransacReprojThreshold=ransac_thresh,
    confidence=confidence,
    maxIters=max_iters,
  )

  if F is None or np.asarray(F).shape != (3, 3):
    return np.zeros((3, 3), dtype=np.float64), len(mkpts0)

  return F.astype(np.float64), len(mkpts0)


### 8.3 Submission CSV Export

Use `generate_submission_csv` to create `submission.csv` from `test.csv`.
- `method='sift_ratio'`: classical path (always available)
- `method='loftr'`: LoFTR path (requires Kornia/Torch)

Set `dry_run=False` and `max_samples=None` for full Kaggle export.


In [ ]:
def load_test_samples(src=SRC):
  test_csv = os.path.join(src, "test.csv")
  if not os.path.exists(test_csv):
    raise FileNotFoundError(f"test.csv not found at: {test_csv}")
  return pd.read_csv(test_csv)


def generate_submission_csv(
  method="sift_ratio",
  out_csv="submission.csv",
  dry_run=True,
  max_samples=25,
  loftr_conf_threshold=0.0,
):
  test_df = load_test_samples(SRC)
  if dry_run and max_samples is not None:
    test_df = test_df.head(max_samples).copy()

  use_loftr = (method == "loftr") and HAS_KORNIA
  loftr_matcher = ensure_loftr_matcher() if use_loftr else None

  rows = []
  stats = []

  for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc=f"Inference ({method})"):
    sample_id = row["sample_id"]
    batch_id = row["batch_id"]
    image_1_id = row["image_1_id"]
    image_2_id = row["image_2_id"]

    p1 = f"{SRC}/test_images/{batch_id}/{image_1_id}.png"
    p2 = f"{SRC}/test_images/{batch_id}/{image_2_id}.png"

    if use_loftr:
      F, n_corr = estimate_f_single_pair_loftr(
        p1,
        p2,
        matcher=loftr_matcher,
        conf_threshold=loftr_conf_threshold,
      )
      stats.append({"sample_id": sample_id, "corr_or_matches": int(n_corr), "method": "loftr"})
    else:
      img1 = cv2.imread(p1)
      img2 = cv2.imread(p2)
      if img1 is None or img2 is None:
        F = np.zeros((3, 3), dtype=np.float64)
        n_corr = 0
      else:
        F, n_corr = estimate_f_single_pair_ratio(img1, img2)
      stats.append({"sample_id": sample_id, "corr_or_matches": int(n_corr), "method": "sift_ratio"})

    rows.append({"sample_id": sample_id, "fundamental_matrix": FlattenMatrix(F)})

  sub_df = pd.DataFrame(rows)
  sub_df.to_csv(out_csv, index=False)

  stat_df = pd.DataFrame(stats)
  print(f"? Wrote {len(sub_df)} rows to: {out_csv}")
  print(f"   Method used: {'loftr' if use_loftr else 'sift_ratio'}")
  if len(stat_df):
    print(f"   Mean correspondences/matches: {stat_df['corr_or_matches'].mean():.2f}")

  return sub_df, stat_df


# Example preview run (safe partial export)
preview_sub_df, preview_stat_df = generate_submission_csv(
  method="loftr" if HAS_KORNIA else "sift_ratio",
  out_csv="submission_preview.csv",
  dry_run=True,
  max_samples=20,
  loftr_conf_threshold=0.0,
)

print("\nPreview submission head:")
print(preview_sub_df.head().to_string(index=False))

print("\nPreview inference stats head:")
print(preview_stat_df.head().to_string(index=False))


## Summary & Pipeline Comparison

This notebook reports **runtime-computed** metrics only:
- Scene-level `mAA`, average matches, and inliers are generated from execution output.
- RANSAC variant comparisons are generated from measured inlier counts and runtime.
- Test-set export supports both classical (SIFT+MAGSAC) and learned (LoFTR+MAGSAC) inference paths.

### Key Takeaways
- Use scene-level `mAA` charts above as the authoritative benchmark for this run.
- Use `generate_submission_csv(...)` for Kaggle-ready `submission.csv` creation.
- No fixed or hardcoded benchmark scores are embedded in this notebook.
